In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# 1. Les og klargjør data
path = Path("data/norway/norwegian.csv")
df = pd.read_csv(path, encoding="utf-8-sig")
df = df.rename(columns=lambda c: c.strip().replace(" ", "_").replace(".", "").replace("%", "pct"))
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
df = df.sort_values("Date").reset_index(drop=True)

In [ ]:
for col in ["Price", "Open", "High", "Low"]:
    df[col] = pd.to_numeric(df[col].str.replace(",", ""), errors="coerce")

df["Change_pct"] = pd.to_numeric(df["Change_pct"].str.replace("%", "").str.replace(",", ""), errors="coerce") / 100
df["Daily_return"] = df["Price"].pct_change()

# 2. Eventanalyse rundt 12. mars 2020
event_date = pd.Timestamp("2020-03-12")
window = 10  # antall handledager på hver side

event_idx = df.index[df["Date"] == event_date][0]
pre = df.iloc[max(event_idx - window, 0):event_idx]
post = df.iloc[event_idx + 1:event_idx + 1 + window]
focus = df.iloc[max(event_idx - window, 0):event_idx + 1 + window]

summary = pd.DataFrame({
    "period": ["pre_window", "event_day", "post_window"],
    "mean_return": [
        pre["Daily_return"].mean(),
        df.at[event_idx, "Daily_return"],
        post["Daily_return"].mean()
    ],
    "cum_return": [
        (1 + pre["Daily_return"]).prod() - 1,
        (1 + df.at[event_idx, "Daily_return"]) - 1,
        (1 + post["Daily_return"]).prod() - 1
    ],
    "avg_price": [
        pre["Price"].mean(),
        df.at[event_idx, "Price"],
        post["Price"].mean()
    ]
})
print(summary)

# 3. Valgfritt: visualiser prisbanen rundt eventet
ax = focus.plot(x="Date", y="Price", marker="o", title="Pris rundt 12. mars 2020")
ax.axvline(event_date, color="red", linestyle="--", label="12. mars 2020")
ax.legend()
plt.tight_layout()
